# 5. DBTL decision benchmark

## Goal

Under a fixed intervention library, candidate pool, and exact-evaluation budget, does the digital twin select better strain–environment designs than conventional routes?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
pool = pd.read_csv(artifact("data/design_benchmark_candidate_pool.csv"))
print("candidate count:", len(pool))
print("candidate fields:", list(pool.columns))
show(pool)


## 2. Data generator

Each candidate explicitly joins an environment with a sparse strain-edit specification. Virtual scoring is a cheap screen; the exact GSM/dynamic oracle is the separate verifier.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# A candidate is the generator input for this phase: explicit environment + edit list.
candidate_columns = [c for c in ["candidate_id", "temperature", "pH", "DO", "edits", "candidate_json"] if c in pool]
show(pool[candidate_columns])


## 3. Model setup and training contract

The learned model produces a control trajectory and virtual predicted outcome. It does not get to change the candidate pool, budget, or exact objective after seeing the answer.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('acceptance', 'data/design_benchmark_acceptance.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

Rank the same declared pool for every method. Record shortlist size, edit mapping, solver calls, and which candidates advance to exact replay.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
acceptance = pd.read_csv(artifact('data/design_benchmark_acceptance.csv'))
show(acceptance)


## 5. Matched comparison

Compare exact verified outcomes under equal budgets—not raw surrogate scores. Acceptance and accounting tables make the comparison auditable.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = acceptance.groupby(['status_type', 'regime_id', 'winner_method', 'status'], dropna=False).size().rename('n').reset_index()
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = acceptance.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

This is a bounded decision-quality result. It says nothing directly about a wet-lab strain until the oracle and observation interface are biologically calibrated.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    raise RuntimeError('Exact DBTL replay is intentionally not a one-line notebook side effect. Use the recorded candidate pool and declared evaluator configuration.')
